# Harici veriyle RGB steganaliz eğitimi (DIV2K, T4 GPU)

**Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU**, sonra **Tümünü çalıştır**.

- DIV2K HR (800 eğitim / 100 doğrulama PNG) Colab içinde indirilir; senin yüklemen gerekmez.
- Her görüntüden 256×256 parçalar kesilir; her gösterimde yeni gizli veri (LSB değiştirme veya ±1 matching, oran %40→%5) gömülür.
- Her epoch sonunda checkpoint Google Drive'a yazılır; bağlantı koparsa defteri yeniden çalıştırınca kaldığı yerden devam eder.
- İsteğe bağlı: Drive'daki `steg-external-rgb/init.pt` varsa eğitim o modelden başlar.
- DIV2K yalnızca akademik araştırma içindir; veri yeniden dağıtılmaz.

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/steg-external-rgb"
os.makedirs(OUT, exist_ok=True)
get_ipython().system("nvidia-smi -L")

In [ ]:
%%bash
set -euo pipefail
if [ ! -d /content/steganography ]; then
  git clone --depth 1 https://github.com/erkanrzgc/steganography /content/steganography
fi
git -C /content/steganography log -1 --format='%h %s'
mkdir -p /content/data && cd /content/data
for part in train valid; do
  if [ ! -d DIV2K_${part}_HR ]; then
    curl -sSL --fail --retry 3 -o DIV2K_${part}_HR.zip https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_${part}_HR.zip
    unzip -q DIV2K_${part}_HR.zip && rm DIV2K_${part}_HR.zip
  fi
done
ls DIV2K_train_HR | wc -l; ls DIV2K_valid_HR | wc -l

In [ ]:
%%bash
set -euo pipefail
cd /content/steganography
[ -f /content/bank/crops.npy ] || python -m research.external_rgb_training bank /content/data/DIV2K_train_HR /content/bank --crops 16
[ -f /content/vbank/crops.npy ] || python -m research.external_rgb_training bank /content/data/DIV2K_valid_HR /content/vbank --crops 2 --seed 2026110906
du -sh /content/bank /content/vbank

In [ ]:
import os
import subprocess

init = os.path.join(OUT, "init.pt")
command = [
    "python", "-m", "research.external_rgb_training", "train",
    "/content/bank", "/content/vbank", OUT,
    "--device", "cuda", "--epochs", "12", "--decay-epoch", "9", "--batch-pairs", "16",
    "--hours", "3.5", "--workers", "2",
]
if os.path.exists(init) and not os.path.exists(os.path.join(OUT, "checkpoint.pt")):
    command += ["--init", init]
print(" ".join(command))
subprocess.run(command, cwd="/content/steganography", check=True)  # noqa: S603

In [ ]:
import json
import os

with open(os.path.join(OUT, "result.json")) as stream:
    result = json.load(stream)
print("durum:", result["status"], "| epoch:", result["epochs"], "| çift/epoch:", result["pairs"])
last = result["history"][-1]["validation"]
print(f"{'koşul':<18}{'AUC':>8}{'doğruluk':>10}")
for key, value in last.items():
    print(f"{key:<18}{value['auc']:>8.3f}{value['accuracy']:>10.3f}")